# Validating the hand-written mixed-model implementation

Before trusting `lmm.py` on the real patient data, we check it
recovers known parameters from simulated data where we control
the truth. Two checks:
1. With near-zero random effects and noise, the fit should reduce
   to (near-exact) ordinary least squares.
2. With real random-effect variance and noise, and a large number
   of simulated groups, the estimates should converge to the true
   generating parameters.

In [1]:
import numpy as np
from lmm import fit_lmm

## Check 1: near-zero noise/random-effects -> should match OLS almost exactly

In [2]:
rng = np.random.default_rng(42)
true_beta = np.array([3.0, -0.02])

y_list, X_list, Z_list = [], [], []
for g in range(20):
    t = np.sort(rng.uniform(20, 130, 3))
    X_i = np.column_stack([np.ones(3), t])
    y_i = X_i @ true_beta + rng.normal(0, 0.001, 3)
    y_list.append(y_i); X_list.append(X_i); Z_list.append(X_i.copy())

fit = fit_lmm(y_list, X_list, Z_list, reml=False, n_restarts=6)
print("True beta:  ", true_beta)
print("Fitted beta:", fit["beta"])
print("Fitted Psi (should be ~0):\n", fit["Psi"])
assert np.allclose(fit["beta"], true_beta, atol=1e-2), "Check 1 FAILED"
print("Check 1 PASSED\n")

True beta:   [ 3.   -0.02]
Fitted beta: [ 2.99970003 -0.01999692]
Fitted Psi (should be ~0):
 [[ 7.27340662e-07 -9.86102407e-09]
 [-9.86102407e-09  1.33692231e-10]]
Check 1 PASSED



## Check 2: real random effects, large n -> estimates should converge to truth

In [3]:
rng = np.random.default_rng(7)
true_beta = np.array([3.0, -0.015])
true_Psi = np.array([[0.04, 0.0005], [0.0005, 0.00002]])
true_sigma2 = 0.01

y_list, X_list, Z_list = [], [], []
for g in range(2000):
    t = np.sort(rng.uniform(20, 130, 3))
    X_i = np.column_stack([np.ones(3), t])
    b_i = rng.multivariate_normal([0, 0], true_Psi)
    y_i = X_i @ (true_beta + b_i) + rng.normal(0, np.sqrt(true_sigma2), 3)
    y_list.append(y_i); X_list.append(X_i); Z_list.append(X_i.copy())

fit = fit_lmm(y_list, X_list, Z_list, reml=True, n_restarts=6)
print("True beta:      ", true_beta, " Fitted:", fit["beta"])
print("True Psi diag:  ", np.diag(true_Psi), " Fitted:", np.diag(fit["Psi"]))
print("True sigma2:    ", true_sigma2, " Fitted:", fit["sigma2"])
assert np.allclose(fit["beta"], true_beta, atol=0.02), "Check 2 FAILED (beta)"
assert np.allclose(np.diag(fit["Psi"]), np.diag(true_Psi), atol=0.01), "Check 2 FAILED (Psi)"
print("Check 2 PASSED")

True beta:       [ 3.    -0.015]  Fitted: [ 3.0022169  -0.01495659]
True Psi diag:   [4.e-02 2.e-05]  Fitted: [4.09926631e-02 1.97391258e-05]
True sigma2:     0.01  Fitted: 0.010005296313045213
Check 2 PASSED
